# Challenge Three: Developing Multi-Agent Systems
### Agentic AI with the Google Agent Development Kit (ADK) — Skills Validation Workshop (15 Points)

**Objectives:**
- Build a multi-agent system consisting of three collaborative agents:
  1. **Coordinating (Root) Agent (`coordinator_agent`)**: The primary entry point that understands user requests, determines intent, and delegates tasks appropriately using the Agent-to-Agent (A2A) protocol.
  2. **Weather Sub-Agent (`weather_agent`)**: Specialized agent equipped with custom tools (`get_lat_lon`, `get_extended_weather_forecast`, `get_weather_alerts`) and lifecycle callbacks for safety and auditing.
  3. **Search Sub-Agent (`search_agent`)**: Specialized agent powered by the ADK built-in Google Search tool (`google_search`) for live web searching, facts, news, and external knowledge.
- Demonstrate task delegation from the root agent to its sub-agents.
- Output event streams in the notebook verifying sub-agent execution and coordination.

## 1. Install Dependencies
Install Google ADK, Vertex AI reasoning engines, and supporting packages.

In [1]:
# In Google Cloud Colab Enterprise, install the required packages
!pip install -q --upgrade \
    "google-cloud-aiplatform[agent_engines,adk]" \
    google-adk \
    requests \
    googlemaps \
    litellm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.6/45.6 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 36.6/36.6 MB 59.8 MB/s eta 0:00:00


## 2. Google Cloud Authentication & Vertex AI Setup (ADC)
Initialize Vertex AI using Application Default Credentials (ADC) in Google Cloud.

In [2]:
import os
import re
import logging
import requests
from typing import Optional, List, Dict, Any, Tuple
from IPython.display import Markdown, display

import google.auth
from google.auth.transport.requests import Request
import vertexai
from vertexai.preview import reasoning_engines
from google.adk.agents import Agent
from google.adk.tools import google_search
from google.adk.agents.callback_context import CallbackContext
from google.adk.models import LlmRequest, LlmResponse

# Set up logging
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger("MultiAgentSystem")

# 1. Resolve Application Default Credentials (ADC)
try:
    credentials, project_id = google.auth.default(
        scopes=["https://www.googleapis.com/auth/cloud-platform", "https://www.googleapis.com/auth/maps-platform"]
    )
    print(f"[Auth] Loaded Application Default Credentials (ADC) for project: {project_id}")
except Exception as e:
    print(f"[Auth Notice] ADC not found ({e}). Ensure 'gcloud auth application-default login' was run.")
    credentials, project_id = None, os.getenv("GOOGLE_CLOUD_PROJECT")

PROJECT_ID = project_id or os.getenv("GOOGLE_CLOUD_PROJECT", "qwicklabs-gcp-01-7a38d08f6f78")
LOCATION = os.getenv("GOOGLE_CLOUD_REGION", "us-central1")

# Instruct Google ADK to route to Vertex AI
os.environ["GOOGLE_GENAI_USE_VERTEXAI"] = "True"
os.environ["GOOGLE_CLOUD_PROJECT"] = PROJECT_ID
os.environ["VERTEXAI_PROJECT"] = PROJECT_ID
os.environ["VERTEXAI_LOCATION"] = LOCATION

if PROJECT_ID != "qwicklabs-gcp-01-7a38d08f6f78":
    vertexai.init(project=PROJECT_ID, location=LOCATION, credentials=credentials)
    print(f"[Vertex AI] Initialized in {LOCATION} for project {PROJECT_ID}.")

[Auth] Loaded Application Default Credentials (ADC) for project: qwiklabs-gcp-01-7a38d08f6f78
[Vertex AI] Initialized in us-central1 for project qwiklabs-gcp-01-7a38d08f6f78.


## 3. Custom Tools for Weather Sub-Agent
Define `get_lat_lon`, `get_extended_weather_forecast`, and `get_weather_alerts`.

In [3]:
def get_adc_access_token() -> Optional[str]:
    """Fetches a fresh Bearer token from ADC for Google Cloud APIs."""
    try:
        creds, _ = google.auth.default(scopes=["https://www.googleapis.com/auth/cloud-platform", "https://www.googleapis.com/auth/maps-platform"])
        if not creds.valid:
            creds.refresh(Request())
        return creds.token
    except Exception:
        return None

US_CITY_COORDINATES_FALLBACK = {
    "miami": {"lat": 25.7617, "lon": -80.1918},
    "seattle": {"lat": 47.6062, "lon": -122.3321},
    "chicago": {"lat": 41.8781, "lon": -87.6298},
    "denver": {"lat": 39.7392, "lon": -104.9903},
    "new york": {"lat": 40.7128, "lon": -74.0060},
    "dallas": {"lat": 32.7767, "lon": -96.7970},
    "san francisco": {"lat": 37.7749, "lon": -122.4194},
    "boston": {"lat": 42.3601, "lon": -71.0589},
}

def get_lat_lon(address: str) -> Optional[Dict[str, float]]:
    """
    Convert a city name, state, or postal address to latitude and longitude coordinates.

    Args:
        address (str): The address or city/state name (e.g., 'Seattle, WA', 'Miami, FL').

    Returns:
        Optional[Dict[str, float]]: Dictionary containing 'lat' and 'lon' floats, or None.
    """
    if not address or not isinstance(address, str):
        return None

    clean_address = address.strip()
    maps_url = "https://maps.googleapis.com/maps/api/geocode/json"

    # 1. ADC Bearer token
    token = get_adc_access_token()
    if token:
        headers = {"Authorization": f"Bearer {token}"}
        if PROJECT_ID and PROJECT_ID != "qwicklabs-gcp-01-7a38d08f6f78":
            headers["X-Goog-User-Project"] = PROJECT_ID
        try:
            resp = requests.get(maps_url, params={"address": clean_address}, headers=headers, timeout=5)
            if resp.status_code == 200:
                data = resp.json()
                if data.get("results"):
                    loc = data["results"][0]["geometry"]["location"]
                    return {"lat": round(float(loc["lat"]), 4), "lon": round(float(loc["lng"]), 4)}
        except Exception:
            pass

    # 2. Maps API Key fallback
    api_key = os.getenv("GOOGLE_MAPS_API_KEY")
    if api_key:
        try:
            resp = requests.get(maps_url, params={"address": clean_address, "key": api_key}, timeout=5)
            if resp.status_code == 200:
                data = resp.json()
                if data.get("results"):
                    loc = data["results"][0]["geometry"]["location"]
                    return {"lat": round(float(loc["lat"]), 4), "lon": round(float(loc["lng"]), 4)}
        except Exception:
            pass

    # 3. Known US cities fallback
    for city_key, coords in US_CITY_COORDINATES_FALLBACK.items():
        if city_key in clean_address.lower():
            return coords
    return None


def get_extended_weather_forecast(lat: float, lon: float) -> Optional[List[Dict[str, str]]]:
    """
    Fetch extended weather forecast from the National Weather Service (NWS) API.

    Args:
        lat (float): Latitude coordinate (e.g., 25.7617).
        lon (float): Longitude coordinate (e.g., -80.1918).

    Returns:
        Optional[List[Dict[str, str]]]: List of forecast period dictionaries, or None.
    """
    headers = {"User-Agent": "(AgenticAI-Workshop, contact@example.com)", "Accept": "application/geo+json"}
    try:
        points_url = f"https://api.weather.gov/points/{round(lat, 4)},{round(lon, 4)}"
        point_res = requests.get(points_url, headers=headers, timeout=8)
        if point_res.status_code != 200:
            return None
        forecast_url = point_res.json()["properties"]["forecast"]
        f_res = requests.get(forecast_url, headers=headers, timeout=8)
        if f_res.status_code != 200:
            return None
        periods = f_res.json()["properties"]["periods"]
        return [
            {
                "name": p.get("name", ""),
                "temperature": f"{p.get('temperature', '')} {p.get('temperatureUnit', '')}",
                "wind": f"{p.get('windSpeed', '')} {p.get('windDirection', '')}",
                "shortForecast": p.get("shortForecast", ""),
                "detailedForecast": p.get("detailedForecast", ""),
            }
            for p in periods[:5]
        ]
    except Exception:
        return None


def get_weather_alerts(lat: float, lon: float) -> Optional[List[Dict[str, str]]]:
    """
    Retrieve active weather warnings, watches, and advisories for coordinates.

    Args:
        lat (float): Latitude coordinate.
        lon (float): Longitude coordinate.

    Returns:
        Optional[List[Dict[str, str]]]: List of active alerts, or empty list if none.
    """
    headers = {"User-Agent": "(AgenticAI-Workshop, contact@example.com)", "Accept": "application/geo+json"}
    try:
        alerts_url = f"https://api.weather.gov/alerts/active?point={round(lat, 4)},{round(lon, 4)}"
        res = requests.get(alerts_url, headers=headers, timeout=8)
        if res.status_code != 200:
            return []
        features = res.json().get("features", [])
        return [
            {
                "event": f["properties"].get("event", "Hazard Alert"),
                "severity": f["properties"].get("severity", "Unknown"),
                "headline": f["properties"].get("headline", ""),
                "description": f["properties"].get("description", "")[:250],
            }
            for f in features
        ]
    except Exception:
        return []

print("[Tools] Registered get_lat_lon, get_extended_weather_forecast, get_weather_alerts.")

[Tools] Registered get_lat_lon, get_extended_weather_forecast, get_weather_alerts.


## 4. Lifecycle Callbacks for Validation & Logging
Implement `before_model_callback` (prompt validation & logging) and `after_model_callback` (response logging).

In [4]:
NON_US_LOCATIONS = [
    "london", "paris", "tokyo", "berlin", "sydney", "rome", "madrid",
    "toronto", "vancouver", "montreal", "beijing", "cairo", "mexico city",
    "mumbai", "dubai", "moscow", "seoul", "singapore", "bangkok", "sao paulo"
]

MALICIOUS_PATTERNS = [
    r"ignore\s+(all\s+)?(previous|prior)\s+instructions",
    r"system\s+prompt",
    r"reveal\s+your\s+(secret|instructions|prompt)",
    r"you\s+are\s+now\s+in\s+developer\s+mode",
    r"dan\s+mode",
    r"drop\s+table",
]

def validate_user_input(callback_context: CallbackContext, llm_request: LlmRequest) -> Optional[LlmResponse]:
    """Validates user prompt for safety and US location constraints."""
    if not llm_request.contents:
        return None
    last = llm_request.contents[-1]
    if last.role != "user" or not last.parts or not getattr(last.parts[0], "text", None):
        return None

    user_text = last.parts[0].text.strip()

    # 1. Malicious prompt injection check
    for pattern in MALICIOUS_PATTERNS:
        if re.search(pattern, user_text, re.IGNORECASE):
            logger.warning("[Security Alert] Blocked prompt injection from user.")
            return LlmResponse(content={
                "role": "model",
                "parts": [{"text": "Request blocked: Message violates security policies or contains unauthorized instructions."}]
            })

    # 2. Location restriction check (only if asking weather for an explicit non-US city)
    lower_text = user_text.lower()
    if any(k in lower_text for k in ["weather", "forecast", "rain", "temperature"]):
        for non_us in NON_US_LOCATIONS:
            if re.search(r"\b" + re.escape(non_us) + r"\b", lower_text):
                logger.info(f"[Location Policy] Blocked non-US weather request: {non_us}")
                return LlmResponse(content={
                    "role": "model",
                    "parts": [{"text": f"Location Error: '{non_us.title()}' is outside the United States. The National Weather Service (NWS) API only supports US locations."}]
                })
    return None


def log_user_prompt(callback_context: CallbackContext, llm_request: LlmRequest) -> Optional[LlmResponse]:
    """Logs user prompts before model execution."""
    if llm_request.contents:
        last = llm_request.contents[-1]
        if last.role == "user" and last.parts and getattr(last.parts[0], "text", None):
            agent_name = getattr(callback_context, "agent_name", "Agent")
            logger.info(f"[{agent_name}] USER >> {last.parts[0].text.strip()}")
    return None


def chained_before_callback(callback_context: CallbackContext, llm_request: LlmRequest) -> Optional[LlmResponse]:
    """Chains validation and prompt logging into a single before_model_callback."""
    val_result = validate_user_input(callback_context, llm_request)
    if val_result is not None:
        return val_result
    log_user_prompt(callback_context, llm_request)
    return None


def log_model_response(callback_context: CallbackContext, llm_response: LlmResponse) -> Optional[LlmResponse]:
    """Logs model responses after generation."""
    if llm_response.content and llm_response.content.parts:
        txt = getattr(llm_response.content.parts[0], "text", None)
        if txt:
            agent_name = getattr(callback_context, "agent_name", "Agent")
            logger.info(f"[{agent_name}] MODEL >> {txt.strip()[:150]}...")
    return None

print("[Callbacks] Chained before_model_callback and log_model_response defined.")

[Callbacks] Chained before_model_callback and log_model_response defined.


## 5. Sub-Agent 1: Specialized Search Agent
Create the Search Sub-Agent using Google ADK's built-in `google_search` tool (`from google.adk.tools import google_search`).

In [5]:
SEARCH_AGENT_INSTRUCTIONS = """You are a specialized Web Search Agent.
Your role is to find accurate, up-to-date, and authoritative information from the internet to answer user questions.

When given an inquiry:
1. Formulate precise search queries using the `google_search` tool.
2. Analyze the search results to extract key facts, dates, news, and relevant context.
3. Synthesize your findings into a clear, concise, and informative answer.
4. Always cite credible sources or references when available.
"""

search_agent = Agent(
    name="search_agent",
    model="gemini-2.5-flash",
    description="Specialist search agent that answers general knowledge questions, facts, news, and current events using Google Search.",
    instruction=SEARCH_AGENT_INSTRUCTIONS,
    tools=[google_search],
)

print(f"[Sub-Agent Created] {search_agent.name} with tool: {search_agent.tools}")

[Sub-Agent Created] search_agent with tool: [<google.adk.tools.google_search_tool.GoogleSearchTool object at 0x7c943502b020>]


## 6. Sub-Agent 2: Specialized Weather Agent
Create the Weather Sub-Agent with NWS tools, geocoding, and lifecycle callbacks.

In [6]:
WEATHER_AGENT_INSTRUCTIONS = """You are the specialized Weather Sub-Agent ("Pat").
Your job is to provide accurate, real-time weather forecasts and severe weather alerts for locations in the United States.

When delegated a weather question:
1. Use `get_lat_lon` with the user's location to obtain the exact coordinates.
2. Use `get_extended_weather_forecast` to fetch current temperatures and upcoming forecast periods from the National Weather Service.
3. Use `get_weather_alerts` to check for active severe weather warnings or advisories.
4. Synthesize the findings into a clear, formatted summary:
   - **Location**: Identified place.
   - **Conditions & Temperatures**: Current and upcoming forecast.
   - **Active Alerts**: Severe warnings or confirm calm conditions.
5. If the location is outside the United States, politely inform that NWS coverage is restricted to the US.
"""

weather_agent = Agent(
    name="weather_agent",
    model="gemini-2.5-flash",
    description="Specialist weather agent for United States real-time forecasts and weather alerts.",
    instruction=WEATHER_AGENT_INSTRUCTIONS,
    tools=[get_lat_lon, get_extended_weather_forecast, get_weather_alerts],
    before_model_callback=chained_before_callback,
    after_model_callback=log_model_response,
)

print(f"[Sub-Agent Created] {weather_agent.name} with tools: {[t.__name__ for t in weather_agent.tools]}")

[Sub-Agent Created] weather_agent with tools: ['get_lat_lon', 'get_extended_weather_forecast', 'get_weather_alerts']


## 7. Coordinating (Root) Agent
Create the Root Coordinator Agent with `weather_agent` and `search_agent` declared in `sub_agents`.
The ADK Agent-to-Agent (A2A) protocol enables the root agent to seamlessly route tasks to sub-agents.

In [7]:
COORDINATOR_INSTRUCTIONS = """You are the primary Coordinating (Root) Agent in a multi-agent system.
Your mission is to understand user requests and delegate work to your specialized sub-agents:

1. `weather_agent`:
   - Dedicated expert for real-time weather forecasts, current temperatures, precipitation, wind conditions, and hazardous weather/storm alerts for locations in the United States.
   - Delegate to `weather_agent` whenever the user asks about weather, forecasts, or alerts.

2. `search_agent`:
   - Dedicated expert for general knowledge, web searches, news, recent events, facts, history, sports, and live information from the internet.
   - Delegate to `search_agent` whenever the user asks about facts, news, definitions, or questions outside weather.

Workflow:
- Determine whether the request is best answered by `weather_agent`, `search_agent`, or both.
- Delegate sub-tasks to the appropriate sub-agent(s).
- Combine and summarize the sub-agents' findings into a cohesive, professional, and helpful response for the user.
"""

root_agent = Agent(
    name="coordinator_agent",
    model="gemini-2.5-flash",
    description="Primary coordinating agent that manages user requests and delegates to weather and search sub-agents.",
    instruction=COORDINATOR_INSTRUCTIONS,
    sub_agents=[weather_agent, search_agent],
)

print(f"[Root Agent Configured] {root_agent.name}")
print(f"  --> Sub-agents: {[sa.name for sa in root_agent.sub_agents]}")

[Root Agent Configured] coordinator_agent
  --> Sub-agents: ['weather_agent', 'search_agent']


## 8. Initialize Reasoning Engines AdkApp & Session
Wrap the multi-agent hierarchy into a `reasoning_engines.AdkApp` and create a persistent session.

In [8]:
# Wrap into AdkApp
app = reasoning_engines.AdkApp(agent=root_agent)

# Create session
user_id = "workshop-student"
session = app.create_session(user_id=user_id)
session_id = session["id"] if isinstance(session, dict) else session.id
print(f"[Session Ready] User: {user_id} | Session ID: {session_id}")

/usr/local/lib/python3.12/dist-packages/vertexai/preview/reasoning_engines/templates/adk.py:1138: UserWarning: [EXPERIMENTAL] InMemoryCredentialService: This feature is experimental and may change or be removed in future versions without notice. It may introduce breaking changes at any time.
  self._tmpl_attrs["credential_service"] = InMemoryCredentialService()
/usr/local/lib/python3.12/dist-packages/google/adk/auth/credential_service/in_memory_credential_service.py:33: UserWarning: [EXPERIMENTAL] BaseCredentialService: This feature is experimental and may change or be removed in future versions without notice. It may introduce breaking changes at any time.
  super().__init__()


[Session Ready] User: workshop-student | Session ID: 03d3c167-a406-4305-97a7-30de99b4842b


## 9. Test Scenarios & Event Output Demonstrating Sub-Agent Delegation
Per Challenge 3, Step 4: *"Write code to test your agents. Output events to your notebook that demonstrate the use of the sub agents."*

We test three scenarios:
1. **Weather Query**: Demonstrates automatic delegation to `weather_agent`.
2. **Search / General Knowledge Query**: Demonstrates delegation to `search_agent`.
3. **Combined Multi-Intent Query**: Demonstrates coordination across both sub-agents.

In [9]:
def run_multi_agent_query(prompt: str, scenario_title: str):
    print("=" * 75)
    print(f"SCENARIO: {scenario_title}")
    print(f"USER QUERY: '{prompt}'")
    print("=" * 75)

    invoked_authors = []
    accumulated_response = ""

    try:
        for event in app.stream_query(user_id=user_id, session_id=session_id, message=prompt):
            author = event.get("author") or event.get("node_info", {}).get("path")
            if author and author not in invoked_authors:
                invoked_authors.append(str(author))
                print(f"  [SUB-AGENT EVENT] Active Agent / Node: {author}")

            if "content" in event and event["content"].get("parts"):
                for part in event["content"]["parts"]:
                    if "text" in part:
                        accumulated_response += part["text"]

        print(f"\n--> Demonstrated Sub-Agent Execution Path: {invoked_authors}")
        print("\n--- FINAL AGENT RESPONSE ---")
        display(Markdown(accumulated_response if accumulated_response else "(Completed)"))

    except Exception as e:
        if "DefaultCredentialsError" in str(e) or "GoogleAuthError" in str(e):
            print(f"[Notice] In offline / local mode without ADC: {e}")
            print("  --> In Google Cloud Colab Enterprise, ADC will authenticate automatically.")
        else:
            print(f"[Query Error]: {e}")


# Scenario 1: Weather Delegation
run_multi_agent_query(
    prompt="What is the current weather forecast and are there any active storm alerts for Miami, Florida?",
    scenario_title="1. WEATHER SUB-AGENT DELEGATION"
)

# Scenario 2: Search Delegation
run_multi_agent_query(
    prompt="What are the latest mission objectives and updates for the NASA Artemis program?",
    scenario_title="2. SEARCH SUB-AGENT DELEGATION"
)

# Scenario 3: Combined Weather and General News Delegation
run_multi_agent_query(
    prompt="What is the weather like in Seattle today, and what are recent news headlines about the city?",
    scenario_title="3. MULTI-AGENT COORDINATED DELEGATION"
)

SCENARIO: 1. WEATHER SUB-AGENT DELEGATION
USER QUERY: 'What is the current weather forecast and are there any active storm alerts for Miami, Florida?'
  [SUB-AGENT EVENT] Active Agent / Node: coordinator_agent
  [SUB-AGENT EVENT] Active Agent / Node: weather_agent

--> Demonstrated Sub-Agent Execution Path: ['coordinator_agent', 'weather_agent']

--- FINAL AGENT RESPONSE ---


Here is the weather forecast and active storm alerts for Miami, Florida:

**Location**: Miami, Florida

**Conditions & Temperatures**:
*   **This Afternoon**: Showers and thunderstorms likely. Mostly sunny, with a high near 82°F. North wind around 26 mph, with gusts as high as 37 mph. Chance of precipitation is 70%. New rainfall amounts between a tenth and quarter of an inch possible.
*   **Tonight**: Showers and thunderstorms likely. Mostly cloudy, with a low around 77°F. Northeast wind 14 to 24 mph, with gusts as high as 33 mph. Chance of precipitation is 70%. New rainfall amounts between a quarter and half of an inch possible.
*   **Wednesday**: A chance of rain showers before 11 am, then a chance of showers and thunderstorms. Mostly sunny, with a high near 84°F. Northeast wind around 14 mph, with gusts as high as 20 mph. Chance of precipitation is 50%. New rainfall amounts less than a tenth of an inch possible.

**Active Alerts**:
*   **Wind Advisory**: issued October 27 at 1:25AM EDT until October 27 at 8:00PM EDT by NWS Miami FL. Northeast winds 15 to 25 mph with gusts up to 40 mph Metro and coastal areas of Miami-Dade, Broward, and Palm Beach counties.
*   **Coastal Flood Warning**: issued October 27 at 1:22AM EDT until October 28 at 8:00PM EDT by NWS Miami FL. Significant coastal flooding expected.
*   **Rip Current Statement**: issued October 27 at 1:22AM EDT until October 28 at 8:00PM EDT by NWS Miami FL. Dangerous rip currents.

SCENARIO: 2. SEARCH SUB-AGENT DELEGATION
USER QUERY: 'What are the latest mission objectives and updates for the NASA Artemis program?'
  [SUB-AGENT EVENT] Active Agent / Node: weather_agent
  [SUB-AGENT EVENT] Active Agent / Node: search_agent

--> Demonstrated Sub-Agent Execution Path: ['weather_agent', 'search_agent']

--- FINAL AGENT RESPONSE ---


The NASA Artemis program aims to return humans to the Moon, establish a sustained lunar presence, and build capabilities for future human missions to Mars. The program involves international and commercial partners such as the European Space Agency (ESA) and SpaceX.

Here are the latest mission objectives and updates for the Artemis program:

**Artemis I (Completed)**
*   Artemis I was an uncrewed test flight of the Space Launch System (SLS) rocket and Orion spacecraft, launched on November 16, 2022.
*   The mission lasted 25 days, testing Orion's critical systems and sending the spacecraft into a Distant Retrograde Orbit (DRO) around the Moon.
*   It also carried 10 CubeSat payloads for research on lunar ice and deep-space radiation effects.
*   The mission successfully concluded on December 11, 2022, with Orion landing in the Pacific Ocean.

**Artemis II (Completed)**
*   Artemis II was the first crewed mission of the Artemis program, launched on April 1, 2026.
*   The mission, lasting ten days, carried four astronauts (Christina Koch, Victor Glover, Reid Wiseman, and Jeremy Hansen) on a trajectory around the Moon to test the Orion spacecraft with a human crew.
*   Objectives included validating ground systems crew rescue operations, testing abort automation, demonstrating Orion's handling qualities and propulsion system upgrades, and validating life support and crew survival systems.
*   The crew also performed meal preparation, waste disposal, atmosphere monitoring, and tested spacesuit performance.
*   The mission aimed to demonstrate systems and operations essential for a crewed lunar campaign.

**Artemis III (Planned for 2027)**
*   Artemis III is scheduled for launch as early as summer 2027 and will be a crewed demonstration flight in low Earth orbit, not a lunar landing.
*   This mission will send four astronauts aboard the Orion spacecraft on an SLS rocket from Kennedy Space Center. The crew includes Randy Bresnik, Luca Parmitano, Frank Rubio, and Andre Douglas.
*   The primary objective is to test rendezvous and docking operations between Orion and test versions of commercial Human Landing Systems (HLS) being developed by Blue Origin and SpaceX.
*   The mission will involve three separate spacecraft: the crewed Orion, and uncrewed test articles of Blue Origin's Blue Moon lander and SpaceX's Starship.
*   The mission will last approximately 14-17 days, during which the crew will enter Blue Moon to validate its subsystems, including environmental controls and life support.
*   Astronauts will also test new extravehicular spacesuits.
*   This mission is designed to reduce risks and gain operational experience before future crewed lunar landings.

**Artemis IV (Planned for 2028)**
*   Artemis IV is scheduled for early 2028 and is now planned to be the first crewed lunar landing of the program since Apollo 17 in 1972.
*   This mission will carry astronauts to lunar orbit, where two crew members will descend to the Moon's South Pole for approximately a week to conduct scientific research.
*   The landing will utilize either SpaceX's Starship HLS or Blue Origin's Blue Moon.
*   The Lunar Gateway project has been canceled, and instead, NASA plans to build a base on the Moon's surface near the lunar south pole.

**Beyond Artemis IV**
*   Beginning with Artemis V, scheduled for no earlier than late 2028, NASA intends to conduct yearly lunar landings to develop a permanent lunar base, as a step toward human missions to Mars.

SCENARIO: 3. MULTI-AGENT COORDINATED DELEGATION
USER QUERY: 'What is the weather like in Seattle today, and what are recent news headlines about the city?'
  [SUB-AGENT EVENT] Active Agent / Node: search_agent

--> Demonstrated Sub-Agent Execution Path: ['search_agent']

--- FINAL AGENT RESPONSE ---


Here's the latest weather forecast for Seattle, Washington, and recent news headlines for the city:

**Current Weather Forecast for Seattle**

As of Tuesday, October 27, 2026, Seattle is experiencing cloudy conditions with a chance of light rain or showers throughout the day. The high temperature is expected to be around 55°F (10-11°C), with an overnight low of approximately 47°F (8°C). There's a possibility of showers continuing into the evening, with some breaks of sun expected. [7, 8, 9, 10, 11]

**Recent News Headlines for Seattle**

*   **Seattle Public Schools Mergers**: Seattle Public Schools are considering mergers as enrollment numbers decline. [3, 4, 5]
*   **City Budget**: Rob Saka has proposed allocating $14 million for traffic safety projects and restoring $1.3 million for pothole repair funding as part of the city's budget review. [5]
*   **Blue Origin Project**: Public records indicate Blue Origin is involved in a "cutting-edge" manufacturing project in Des Moines, a city within the Seattle metropolitan area. [4, 6]
*   **Regional Traffic**: All lanes of US 2 recently reopened after a multi-vehicle collision near Monroe caused hours-long closures. [3, 4, 6]
*   **Winter Weather Advisory**: A winter weather advisory has been issued for the Cascade Mountains, signaling upcoming winter travel conditions. [3, 4, 9]
*   **Shooting Incident**: In July, a shooting at a Seattle food festival resulted in three fatalities and four injuries. A 15-year-old was detained in connection with the incident, and authorities are reportedly still searching for a second suspect. [2]

## 10. Summary & Verification
✅ **Coordinating (Root) Agent**: `coordinator_agent` orchestrates user queries.
✅ **Weather Sub-Agent**: `weather_agent` with NWS tools and prompt security/location callbacks.
✅ **Search Sub-Agent**: `search_agent` with ADK built-in `google_search` tool.
✅ **A2A Protocol & Event Streams**: Sub-agent execution demonstrated across real queries.